## Scalable DMRG (`tensor_network_updates`)

The original `pstrings_to_mpo_optimized` builds the *exact* MPO (bond dimension
= number of terms `K`) via dense `(2,2,K,K)` diagonal tensors before truncating,
so it needs `O(K**2)` memory and dies for large `K`.

`tensor_network_updates` instead does a single left-to-right compression sweep
that never forms the full `K`-bond: memory is `O(max_bond * K)` and time is
`O(n_qubits * K * max_bond**2)`. This scales to ~40 qubits and
hundreds-of-thousands to millions of terms.

### Two different bond dimensions (important for chemistry!)

There are **two** independent bond dimensions in this workflow, and they do
very different things:

1. **MPO `max_bond`** (in `mpo_from_pauliwordop`) compresses the *Hamiltonian
   operator*. Truncating it is **not variational** — the DMRG energy of a
   truncated MPO can even fall *below* the true ground-state energy, and no
   amount of DMRG sweeps or state bond dimension can fix a corrupted operator.
   For chemistry Hamiltonians the *exact* MPO bond dimension is modest (grows
   polynomially), so build it essentially exactly:
   `max_bond=None`, small `cutoff`, `dtype=np.complex128`.

2. **DMRG `bond_dims`** (in `find_groundstate_quimb`) is the *state* bond
   dimension. Once the MPO is (near-)exact, **this** is the knob that controls
   accuracy — increase it to converge the energy from above to FCI.

The `max_bond=100`, `complex64` settings below are only to keep the 40-qubit
*random* demo cheap. If your energy "won't improve with bond dimension", you are
almost certainly truncating the MPO operator: set `max_bond=None` and
`dtype=np.complex128` and vary `bond_dims` instead.

In [23]:
from symmer.operators import PauliwordOp
from symmer.approximate.tensor_network_updates import (
    mpo_from_pauliwordop,
    find_groundstate_quimb,
)
import numpy as np

# random H

In [24]:
# large qubit count + many terms
Hq = PauliwordOp.random(n_qubits=40, n_terms=200_000, complex_coeffs=False, density=0.6)

## add a dominant term to make ground state easy to find... so looking at
## bitstrings is more interesting
Hq += PauliwordOp.from_list([Hq.n_qubits * "Z"], [10_000])


# PauliwordOp -> compressed MPO (bounded bond dimension, complex64 to save memory)
MPO = mpo_from_pauliwordop(Hq, 
                            max_bond=100,
                             cutoff=1e-10, 
                             dtype=np.complex64,
                            
                            #  method = "svd" # default (more accurate)
                             method = "gram" # low memory (less accurate)
                             )

In [25]:
# solve for the ground-state energy with DMRG.
# For 40 qubits do NOT densify the state (2**40 amplitudes); keep the MPS.
result = find_groundstate_quimb(
    MPO,
    bond_dims=[150, 150, 150, 200],
    cutoffs=1e-8,
    max_sweeps=8,
    verbosity=1,
)
print("ground-state energy:", result["energy"])

1, R, max_bond=(150/150), cutoff:1e-08


100%|###########################################| 39/39 [00:00<00:00, 98.66it/s]

Energy: (-10000.0087890625-0.0005960464477539062j) ... not converged.
2, R, max_bond=(32/150), cutoff:1e-08



100%|#########################################| 39/39 [00:00<00:00, 1624.73it/s]

Energy: (-10000.0078125+0.0004214684886392206j) ... not converged.
3, R, max_bond=(8/150), cutoff:1e-08



100%|#########################################| 39/39 [00:00<00:00, 1869.33it/s]

Energy: (-9999.9990234375+0j) ... not converged.
4, R, max_bond=(1/200), cutoff:1e-08



100%|#########################################| 39/39 [00:00<00:00, 2039.75it/s]

Energy: (-9999.9990234375+0j) ... converged!
ground-state energy: -9999.9990234375


In [26]:
result["mps"]

MatrixProductState(tensors=40, indices=79, L=40, max_bond=1)

# chemistry example

In [27]:
import json
import os
symmer_dir = os.path.dirname(os.path.dirname(os.getcwd()))
hamiltonian_data_dir = os.path.join(symmer_dir, "tests", "hamiltonian_data")

# H_file = os.path.join(hamiltonian_data_dir, "F2_STO-3G_SINGLET_JW.json")
H_file = os.path.join(hamiltonian_data_dir, "H3O+_STO-3G_SINGLET_JW.json")

with open(H_file, "r") as infile:
    H_data = json.load(infile)
    
Hq = PauliwordOp.from_dictionary(H_data["hamiltonian"])
HF_E  = H_data['data']['calculated_properties']["HF"]
FCI_E = H_data['data']['calculated_properties']["FCI"]

print("Hq number of qubits    :", Hq.n_qubits)
print("Hq number of Pauli ops :", Hq.n_terms)

Hq number of qubits    : 16
Hq number of Pauli ops : 2469


In [28]:
# PauliwordOp -> compressed MPO (bounded bond dimension, complex64 to save memory)
## NOTE: this does not account for any molecular symmetries!
## need to add constraints for number, spin and group symmetries
MPO = mpo_from_pauliwordop(Hq, 
                            max_bond=200, # <- accuracy of MPO 
                            cutoff=1e-10, 
                            dtype=np.complex64,
                            
                            method = "svd", # default (more accurate)
                            #  method = "gram", # low memory (less accurate)
                             )
MPO

MatrixProductOperator(tensors=16, indices=47, L=16, max_bond=200)

In [29]:
from symmer.approximate.tensor_network_updates import computational_basis_mps
hf_mps = computational_basis_mps(H_data['data']['hf_array'])   # bond-1 MPS
hf_mps

MatrixProductState(tensors=16, indices=31, L=16, max_bond=1)

In [30]:
## get <mps|H|mps>
ref_energy = (hf_mps.H @ MPO.apply(hf_mps)).real
ref_energy

np.float64(-75.33042108652467)

In [31]:
# solve for the ground-state energy with DMRG.
# NOTE: this does not account for any molecular symmetries!
#       need to add constraints for number, spin and group symmetries
result = find_groundstate_quimb(
    MPO,
    bond_dims=[200, 200, 200, 200],
    cutoffs=1e-8,
    max_sweeps=8,
    verbosity=1,

    p0=hf_mps #<-- optional initial guess mps
    # p0=None
)
print("ground-state energy:", result["energy"])
print("FCI energy:", FCI_E["energy"])

1, R, max_bond=(1/200), cutoff:1e-08


100%|##########################################| 15/15 [00:00<00:00, 431.62it/s]

Energy: (-75.3304210865247+2.3570657264882138e-07j) ... not converged.
2, R, max_bond=(1/200), cutoff:1e-08



100%|##########################################| 15/15 [00:00<00:00, 511.19it/s]

Energy: (-75.33042108652464+2.357065693181524e-07j) ... converged!
ground-state energy: -75.33042108652464
FCI energy: -75.3936259005811


### Reading off the highest-amplitude bitstrings

The ground state is an MPS, so the full `2**n` vector is never formed. To find
the most-probable computational-basis bitstrings we search the MPS directly:

* `method='beam'` (default) — beam search over the most-probable prefixes.
  Always fast (`O(n * beam_width * D**2)`) and terminating; exact for peaked
  states (increase `beam_width` for more accuracy).
* `method='exact'` — best-first branch-and-bound with the admissible bound
  `sqrt(P(prefix))`; provably returns the true top-`k`, but is only efficient
  for peaked / low-entanglement states.

`bitstring[i]` is the value of qubit `i` (same ordering as the Pauli strings).

In [32]:
from symmer.approximate.tensor_network_updates import top_amplitude_bitstrings

# top-k most probable bitstrings straight from the MPS (no dense state)
top = top_amplitude_bitstrings(result["mps"], k=10, beam_width=512)

print(f"{'probability':>14}   {'amplitude':>22}   bitstring")
for bitstring, amp, prob in top:
    print(f"{prob:14.3e}   {amp:22.5f}   {bitstring}")

   probability                amplitude   bitstring
     1.000e+00         0.61728-0.78675j   1111111111000000
     1.903e-18         0.00000+0.00000j   1111101111000000
     2.973e-19         0.00000+0.00000j   1111111111000001
     4.358e-25         0.00000-0.00000j   1111111111100000
     7.219e-27         0.00000-0.00000j   1111111101000000
     2.110e-27        -0.00000+0.00000j   1101111111000000
     1.861e-27         0.00000-0.00000j   1111110111000000
     5.553e-28        -0.00000+0.00000j   1111111011000000
     4.129e-28        -0.00000+0.00000j   1111111111001000
     3.085e-28        -0.00000+0.00000j   1111111111000010
